In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency,f_oneway
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#会话聚合
df = pd.read_csv('events_clean.csv')
session_agg = df.groupby(
    ['session_id','customer_id','experiment_group']
).agg(
    session_duration=("session_duration_sec", "max"),
    is_bounce=("event_type", lambda x: 1 if "bounce" in x.values else 0),
    has_add_to_cart=("event_type", lambda x: 1 if "add_to_cart" in x.values else 0),
    has_purchase=("event_type", lambda x: 1 if "purchase" in x.values else 0)
).reset_index()

In [ ]:
#分组指标汇总
group_metrics = session_agg.groupby("experiment_group").agg(
    session_count=("session_id", "size"),
    bounce_sessions=("is_bounce", "sum"),
    addcart_sessions=("has_add_to_cart", "sum"),
    purchase_sessions=("has_purchase", "sum"),
    avg_session_duration=("session_duration", "mean")
).reset_index()

# 计算转化率
group_metrics["bounce_rate"] = group_metrics["bounce_sessions"] / group_metrics["session_count"]
group_metrics["add_to_cart_rate"] = group_metrics["addcart_sessions"] / group_metrics["session_count"]
group_metrics["purchase_rate"] = group_metrics["purchase_sessions"] / group_metrics["session_count"]

print("\n===== 各组核心指标汇总 =====")
print(group_metrics.round(4))

In [ ]:
#显著性检验
#购买转化率
ct_purchase = pd.crosstab(session_agg["experiment_group"], session_agg["has_purchase"])
chi2_p, p_purchase, dof, exp = chi2_contingency(ct_purchase)
print(f"\n【购买转化率 卡方检验】p值 = {p_purchase:.4f}")

# 加购转化率：卡方检验
ct_addcart = pd.crosstab(session_agg["experiment_group"], session_agg["has_add_to_cart"])
chi2_add, p_addcart, dof, exp = chi2_contingency(ct_addcart)
print(f"【加购转化率 卡方检验】p值 = {p_addcart:.4f}")

# 会话时长：单因素方差分析ANOVA
ctrl = session_agg[session_agg["experiment_group"] == "Control"]["session_duration"]
va = session_agg[session_agg["experiment_group"] == "Variant_A"]["session_duration"]
vb = session_agg[session_agg["experiment_group"] == "Variant_B"]["session_duration"]
f_stat, p_duration = f_oneway(ctrl, va, vb)
print(f"【平均会话时长 ANOVA检验】p值 = {p_duration:.4f}")

# ====================== 5. 结果解读模板 ======================
alpha = 0.05
print("\n===== 结论参考（α=0.05） =====")
if p_purchase < alpha:
    print("三组购买转化率存在统计学显著差异，需要进一步两两对比")
else:
    print("没有充分证据证明三组购买转化率存在显著差异")

if p_addcart < alpha:
    print("三组加购转化率存在统计学显著差异，需要进一步两两对比")
else:
    print("没有充分证据证明三组加购转化率存在显著差异")

if p_duration < alpha:
    print("三组平均会话时长存在统计学显著差异，需要进一步两两对比")
else:
    print("没有充分证据证明三组平均会话时长存在显著差异")


In [ ]:
#  显著性检验
# 购买转化率：卡方检验（多组二分类指标）
ct_purchase = pd.crosstab(session_agg["experiment_group"], session_agg["has_purchase"])
chi2_p, p_purchase, dof, exp = chi2_contingency(ct_purchase)
print(f"\n【购买转化率 卡方检验】p值 = {p_purchase:.4f}")

#chi2_p:卡方统计量   p_purchase:p值  dof:自由度  exp：理论期望频数
# 加购转化率：卡方检验
ct_addcart = pd.crosstab(session_agg["experiment_group"], session_agg["has_add_to_cart"])
chi2_add, p_addcart, dof, exp = chi2_contingency(ct_addcart)
print(f"【加购转化率 卡方检验】p值 = {p_addcart:.4f}")

# 会话时长：单因素方差分析ANOVA
ctrl = session_agg[session_agg["experiment_group"] == "Control"]["session_duration"]
va = session_agg[session_agg["experiment_group"] == "Variant_A"]["session_duration"]
vb = session_agg[session_agg["experiment_group"] == "Variant_B"]["session_duration"]
f_stat, p_duration = f_oneway(ctrl, va, vb)
print(f"【平均会话时长 ANOVA检验】p值 = {p_duration:.4f}")


alpha = 0.05

from statsmodels.stats.proportion import proportions_ztest

def two_group_prop_test(df, group1, group2, target_col):
    g1 = df[df["experiment_group"] == group1]
    g2 = df[df["experiment_group"] == group2]
    count = [g1[target_col].sum(), g2[target_col].sum()]
    nobs = [len(g1), len(g2)]
    stat, p = proportions_ztest(count, nobs)
    return stat, p

pairs = [("Control","Variant_A"),("Control","Variant_B"),("Variant_A","Variant_B")]
alpha_corrected = 0.05 / 3
print(f"\n===== 两两对比（Bonferroni校正阈值 α'={alpha_corrected:.4f}） =====")

# 购买转化率两两检验
print("\n【购买转化率两两Z检验】")
for g1,g2 in pairs:
    _,p = two_group_prop_test(session_agg, g1,g2,"has_purchase")
    sig = "显著" if p < alpha_corrected else "不显著"
    print(f"{g1} vs {g2}: p={p:.4f} | {sig}")

# 加购转化率两两检验
print("\n【加购转化率两两Z检验】")
for g1,g2 in pairs:
    _,p = two_group_prop_test(session_agg, g1,g2,"has_add_to_cart")
    sig = "显著" if p < alpha_corrected else "不显著"
    print(f"{g1} vs {g2}: p={p:.4f} | {sig}")
